# 🏆 Capstone Project: **Reference Solution**
### AgriReady: from raw farm records to machine-learning-ready data

> ⚠️ **Try the project yourself first** (`project_template.ipynb`). This notebook is *one* good solution: not the only one. Compare approaches *after* you have your own.

All deliverables are written next to this notebook: `farm_clean.csv`, `preprocessing_pipeline.joblib`, `yield_model_pipeline.joblib`, the ML-ready matrices, `feature_dictionary.csv` and `TRANSFORMATION_LOG.md`.

In [1]:
import sys
sys.path.append("../../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

---
# Step 0 · Frame the problem (10 minutes, but the most important 10 minutes)

Before touching data, write down **what the model will be used for and when**. Everything in your pipeline depends on it.

> **Use-case for this project: a pre-harvest yield forecast.** *Shortly before harvest, the county office wants to forecast each farm's yield (t/ha) so it can plan storage and transport.*
> At that moment the planting date, soil tests, farm attributes, inputs used, and the season's weather and pest pressure are **known**. The actual harvest date, the sale price and the revenue are **not**.

**Your task in this cell:** run it and read the setup. Then write 3–4 sentences stating the prediction moment in *your own words* in the **Prediction-moment statement** cell at the top of the notebook.

In [2]:
from pathlib import Path
import re, difflib, joblib
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

OUT = Path(".")                       # deliverables are written here
print("Setup complete. Deliverables will be saved in:", OUT.resolve())

Setup complete. Deliverables will be saved in: /tmp/ctest/project/reference_solution


**✍️ Prediction-moment statement** *(write yours in the template; this is the reference answer)*

> We forecast yield shortly **before harvest**. Allowed inputs: farm attributes, planting date, soil tests, inputs applied (fertiliser, irrigation), crop, and the season's weather and pest pressure. **Excluded:** the actual harvest date (the harvest hasn't happened), market price and total revenue (they only exist after harvest and sale, and revenue is literally computed from the target), and `farm_id` (an identifier, not a driver). We therefore evaluate on **unseen farms** (grouped split), because the model will be used on farms it has not seen.

---
# Step 1 · Audit the raw data  *(Lessons 0–2)*

**Goal:** an *issues log*: a written, evidence-backed list of everything wrong with the raw file, **before** you fix anything.

**Do this:**
1. Load the raw data and run `profile()`.
2. For each issue family below, write code that *counts* the problem:
   - duplicates (exact and near),
   - wrong types (numbers/dates stored as text),
   - inconsistent category spellings,
   - impossible values (use agronomic common sense),
   - missing values (how much, where, and is it random?),
   - suspicious outliers.
3. Collect the counts in a DataFrame called `issues`.

**Deliverable:** the `issues` table (you will reuse it in the final documentation).

In [3]:
raw = load_raw_farm()
display(profile(raw))

issues = pd.Series({
    "rows": len(raw),
    "exact duplicate rows": int(raw.duplicated().sum()),
    "distinct crop spellings (real: 5)": raw["crop"].nunique(),
    "distinct county spellings (real: 10)": raw["county"].nunique(),
    "distinct irrigation codes (real: 2)": raw["irrigation"].nunique(),
    "fertilizer entries that are not plain numbers": int((~raw["fertilizer_kg_ha"].dropna().str.fullmatch(r"\d+(\.\d+)?")).sum()),
    "dates not in ISO yyyy-mm-dd format": int((~raw["planting_date"].dropna().str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()),
    "impossible soil_ph (<3 or >10)": int(((raw["soil_ph"] < 3) | (raw["soil_ph"] > 10)).sum()),
    "impossible rainfall (<=0)": int((raw["rainfall_mm"] <= 0).sum()),
    "temperature sentinel codes (-999 / 99)": int(raw["avg_temp_c"].isin([-999, 99]).sum()),
    "non-positive farm size": int((raw["farm_size_ha"] <= 0).sum()),
    "farm size > 80 ha (outlier candidates)": int((raw["farm_size_ha"] > 80).sum()),
    "fertilizer > 1000 kg/ha (after text extraction)": int((pd.to_numeric(raw["fertilizer_kg_ha"].str.extract(r"(\d+\.?\d*)", expand=False)) > 1000).sum()),
    "negative or missing target (yield)": int((raw["yield_t_ha"] <= 0).sum() + raw["yield_t_ha"].isna().sum()),
}, name="count").to_frame()
issues

,dtype,n_missing,pct_missing,n_unique,example
farm_id,str,0,0.0,600,F0431
county,str,0,0.0,37,Machakos
soil_type,str,75,3.0,16,Clay
farm_size_ha,float64,49,2.0,298,1.06
experience,str,127,5.1,4,Advanced
crop,str,0,0.0,27,Potato
season,str,50,2.0,2,Long rains
planting_date,str,0,0.0,631,2023-04-06
harvest_date,str,0,0.0,1148,2023-07-18
rainfall_mm,float64,99,4.0,1934,491.1


,count
rows,2485
exact duplicate rows,60
distinct crop spellings (real: 5),27
distinct county spellings (real: 10),37
distinct irrigation codes (real: 2),10
fertilizer entries that are not plain numbers,382
dates not in ISO yyyy-mm-dd format,107
impossible soil_ph (<3 or >10),10
impossible rainfall (<=0),7
temperature sentinel codes (-999 / 99),5


Missing values deserve their own look: *how much, and is it random?*

In [4]:
miss = (raw.isna().mean() * 100).round(1).sort_values(ascending=False)
display(miss[miss > 0].to_frame("% missing"))

# Is fertilizer missingness related to farm size? (MAR check from Lesson 1)
small = raw["farm_size_ha"] < 1
print("fertilizer missing | farms < 1 ha :", round(raw.loc[small, "fertilizer_kg_ha"].isna().mean() * 100, 1), "%")
print("fertilizer missing | farms >= 1 ha:", round(raw.loc[~small, "fertilizer_kg_ha"].isna().mean() * 100, 1), "%")

,% missing
fertilizer_kg_ha,10.5
organic_matter_pct,6.9
soil_ph,6.0
experience,5.1
nitrogen_ppm,5.0
phosphorus_ppm,5.0
potassium_ppm,4.9
pest_pressure,4.0
rainfall_mm,4.0
avg_temp_c,3.1


fertilizer missing | farms < 1 ha : 20.4 %
fertilizer missing | farms >= 1 ha: 1.7 %


---
# Step 2 · Clean the data: `clean_farm()`  *(Lessons 1–2)*

**Goal:** a function `clean_farm(raw) -> DataFrame` that performs all **structural** cleaning and learns *nothing* from the data (no means, no medians, no IQR fences; those belong in the pipeline).

**Your function must (in this order):**
1. Convert types: fertilizer → numeric (extract the number), dates → datetime (`format='mixed'`).
2. Standardise `crop`, `county`, `soil_type`, `pest_pressure`; convert `irrigation` to 0/1.
3. Remove duplicates on the natural key `farm_id + planting_date`, keeping the most complete record.
4. Cross-field logic: derive missing `season` from the planting month; blank a harvest date that precedes planting.
5. Invalid values → NaN using a **rule table** (pH 3–10, rainfall 50–3000, temperature −5–40, farm size > 0, fertilizer ≤ 1000, …).
6. Target: drop rows whose yield is missing, ≤ 0, or above the crop's physical ceiling (Maize 12, Beans 4, Wheat 10, Potato 45, Sorghum 9 t/ha).
7. Return a cleaning **log** (how many rows/values each step touched).

**Self-check:** `check_clean(farm_clean)` must pass 13/13.

> 💡 You may re-use ideas and code from Lesson 02, but *understand* every line. Don't copy blindly.

In [5]:
CROPS    = ["Maize", "Beans", "Wheat", "Potato", "Sorghum"]
COUNTIES = ["Uasin Gishu", "Trans Nzoia", "Nakuru", "Kakamega", "Bungoma", "Meru", "Machakos", "Nyeri", "Kisii", "Kiambu"]
SOILS    = ["Loam", "Clay", "Sandy", "Silt", "Volcanic"]
PESTS    = ["Low", "Medium", "High"]
RULES = {"soil_ph": (3, 10), "rainfall_mm": (50, 3000), "avg_temp_c": (-5, 40), "farm_size_ha": (0.01, 1000),
         "nitrogen_ppm": (0, 500), "phosphorus_ppm": (0, 300), "potassium_ppm": (0, 800), "organic_matter_pct": (0, 30),
         "fertilizer_kg_ha": (0, 1000), "market_price_ksh_kg": (1, 1000)}
YIELD_MAX = {"Maize": 12, "Beans": 4, "Wheat": 10, "Potato": 45, "Sorghum": 9}


def standardise(series, canonical, synonyms=None, cutoff=0.8):
    """normalise -> synonyms -> fuzzy match -> unmatched becomes NaN"""
    norm = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
    lookup = {norm(c): c for c in canonical}
    lookup.update({norm(k): v for k, v in (synonyms or {}).items()})
    keys = list(lookup)
    def one(x):
        if pd.isna(x):
            return np.nan
        k = norm(x)
        if k in lookup:
            return lookup[k]
        close = difflib.get_close_matches(k, keys, n=1, cutoff=cutoff)
        return lookup[close[0]] if close else np.nan
    return series.map(one)


def to_flag(series):
    YES, NO = {"yes", "y", "true", "1"}, {"no", "n", "false", "0"}
    k = series.astype(str).str.strip().str.lower()
    return k.map(lambda v: 1 if v in YES else 0 if v in NO else np.nan)


def clean_farm(raw, verbose=True):
    """Structural cleaning only - learns nothing from the data."""
    log = {"raw rows": len(raw)}
    df = raw.copy()

    # 1. types
    df["fertilizer_kg_ha"] = pd.to_numeric(df["fertilizer_kg_ha"].str.extract(r"(\d+\.?\d*)", expand=False))
    for c in ["planting_date", "harvest_date"]:
        df[c] = pd.to_datetime(df[c], format="mixed")

    # 2. consistent categories (BEFORE de-duplication)
    df["crop"] = standardise(df["crop"], CROPS, {"corn": "Maize", "irish potato": "Potato"})
    df["county"] = standardise(df["county"], COUNTIES)
    df["soil_type"] = standardise(df["soil_type"], SOILS)
    df["pest_pressure"] = standardise(df["pest_pressure"], PESTS, {"med": "Medium"})
    df["irrigation"] = to_flag(df["irrigation"])

    # 3. duplicates on the natural key, keep the most complete record
    n = len(df)
    df = (df.assign(_n_missing=df.isna().sum(axis=1)).sort_values("_n_missing")
            .drop_duplicates(subset=["farm_id", "planting_date"], keep="first").drop(columns="_n_missing").sort_index())
    log["duplicate records removed"] = n - len(df)

    # 4. cross-field logic
    month = df["planting_date"].dt.month
    derived = np.where(month.isin([3, 4, 5]), "Long rains", np.where(month.isin([10, 11, 12]), "Short rains", None))
    log["missing 'season' derived from planting month"] = int(df["season"].isna().sum())
    df["season"] = df["season"].fillna(pd.Series(derived, index=df.index))
    bad = df["harvest_date"] < df["planting_date"]
    log["harvest dates blanked (before planting)"] = int(bad.sum())
    df.loc[bad, "harvest_date"] = pd.NaT

    # 5. impossible values -> NaN
    before = int(df[list(RULES)].isna().sum().sum())
    for col, (lo, hi) in RULES.items():
        df[col] = df[col].where(df[col].between(lo, hi))
    log["impossible feature values set to NaN"] = int(df[list(RULES)].isna().sum().sum()) - before

    # 6. target rows
    n = len(df)
    ok = df["yield_t_ha"].notna() & (df["yield_t_ha"] > 0) & (df["yield_t_ha"] <= df["crop"].map(YIELD_MAX))
    df = df.loc[ok].reset_index(drop=True)
    log["rows dropped (missing / impossible yield)"] = n - len(df)
    log["clean rows"] = len(df)

    if verbose:
        print("CLEANING LOG")
        for k, v in log.items():
            print(f"  {k:50s} {v:>6}")
    df.attrs["cleaning_log"] = log
    return df

Run your function, run the self-check and save the first deliverable.

In [6]:
farm_clean = clean_farm(load_raw_farm())
print()
check_clean(farm_clean)
farm_clean.to_csv(OUT / "farm_clean.csv", index=False)
print("\nsaved:", OUT / "farm_clean.csv", farm_clean.shape)

CLEANING LOG
  raw rows                                             2485
  duplicate records removed                              85
  missing 'season' derived from planting month           48
  harvest dates blanked (before planting)                14
  impossible feature values set to NaN                   35
  rows dropped (missing / impossible yield)              71
  clean rows                                           2329

  ✅ no exact duplicate rows
  ✅ one record per farm_id + planting_date
  ✅ planting_date / harvest_date are datetimes
  ✅ fertilizer_kg_ha is numeric
  ✅ exactly 5 crop labels
  ✅ exactly 10 county labels
  ✅ irrigation is 0/1
  ✅ soil_ph within 3-10
  ✅ rainfall within 50-3000 mm
  ✅ avg_temp within -5..40 C
  ✅ farm_size_ha > 0
  ✅ target has no NaN and is positive
  ✅ harvest after planting (where present)

13/13 checks passed

saved: farm_clean.csv (2329, 22)


---
# Step 3 · Decide what the model may see  *(Lesson 6)*

**Goal:** a dictionary `EXCLUDE` mapping every column you remove to a one-line **reason**, then the objects `X`, `y` (log-yield) and `groups` (farm ids).

Use the *timeline test* from Lesson 6: "would I know this at the prediction moment you wrote in Step 0?" Consider each column. Watch out for: post-harvest columns, columns derived from the target, identifiers, and columns that duplicate other information.

In [7]:
EXCLUDE = {
    "farm_id":             "identifier - would let the model memorise farms instead of learning agronomy",
    "harvest_date":        "actual harvest has not happened at prediction time (grow_days would leak the future)",
    "total_revenue_ksh":   "computed from the target (yield x area x price) - pure target leakage",
    "market_price_ksh_kg": "only known at sale, after harvest",
    "season":              "redundant: re-derived from the planting date inside the pipeline (rain_season)",
}
X = farm_clean.drop(columns=["yield_t_ha"] + list(EXCLUDE))
y = np.log(farm_clean["yield_t_ha"])              # model log(yield): errors are proportional, skew is reduced
groups = farm_clean["farm_id"]
print("candidate predictors:", X.shape[1], "->", list(X.columns))
pd.Series(EXCLUDE, name="reason").to_frame()

candidate predictors: 16 -> ['county', 'soil_type', 'farm_size_ha', 'experience', 'crop', 'planting_date', 'rainfall_mm', 'avg_temp_c', 'soil_ph', 'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm', 'organic_matter_pct', 'fertilizer_kg_ha', 'irrigation', 'pest_pressure']


,reason
farm_id,identifier - would let the model memorise farm...
harvest_date,actual harvest has not happened at prediction ...
total_revenue_ksh,computed from the target (yield x area x price...
market_price_ksh_kg,"only known at sale, after harvest"
season,redundant: re-derived from the planting date i...


---
# Step 4 · Split first!  *(Lesson 6)*

**Goal:** a hold-out **test set** of unseen **farms** (20%), locked away until Step 10. Use `GroupShuffleSplit`.

**Rule from here on:** nothing is ever `fit` on `X_test`.

In [8]:
tr, te = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(X, groups=groups))
X_train, X_test = X.iloc[tr], X.iloc[te]
y_train, y_test = y.iloc[tr], y.iloc[te]
groups_train = groups.iloc[tr]

assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])), "a farm appears in both sets!"
print(f"train: {len(X_train)} rows / {groups.iloc[tr].nunique()} farms   |   test: {len(X_test)} rows / {groups.iloc[te].nunique()} farms")

train: 1866 rows / 480 farms   |   test: 463 rows / 120 farms


---
# Step 5 · Engineer features  *(Lesson 4)*

**Goal:** at least **8 new features**, covering all of these families, each justified in one sentence:

| Family | Minimum |
|---|---|
| Date-based | 3 (e.g. year, month, sin/cos of month, day of year, rain season) |
| Combination | 3 (ratio, sum, index, interaction, e.g. N:P ratio, NPK total, moisture index) |
| Domain knowledge | 2 (e.g. squared distance from the crop's optimal rainfall/temperature, pH gap) |
| Transform | 1 (e.g. log of farm size) |

**Implementation requirement:** write them as **scikit-learn transformers** (classes with `fit` and `transform`) so they work inside the pipeline. Row-wise formulas learn nothing (`fit` just returns `self`).

You may peek at `utils/agri_prep.py` *after* you've tried: the reference solution uses it.

**Also remember:** time-aware features (rolling/lag) are only valid when built from the **past**. If you attempt the stretch goal (price context from `crop_prices_daily.csv`), use trailing windows.

In [9]:
from agri_prep import DateFeatures, AgriFeatures, Winsorizer

eng = AgriFeatures().fit_transform(DateFeatures(harvest_col=None).fit_transform(X_train))
new_cols = [c for c in eng.columns if c not in X_train.columns]
print(len(new_cols), "engineered features:", new_cols)
eng[new_cols].head(3).round(2)

14 engineered features: ['plant_year', 'plant_month', 'plant_month_sin', 'plant_month_cos', 'plant_dayofyear', 'rain_season', 'log_farm_size', 'npk_total', 'n_to_p_ratio', 'moisture_index', 'fert_x_irrigation', 'rain_gap_sq', 'temp_gap_sq', 'ph_gap_sq']


,plant_year,plant_month,plant_month_sin,plant_month_cos,plant_dayofyear,rain_season,log_farm_size,npk_total,n_to_p_ratio,moisture_index,fert_x_irrigation,rain_gap_sq,temp_gap_sq,ph_gap_sq
0,2023,4,0.87,-0.50,96,Long,0.72,265.0,2.30,22.63,66.0,11859.21,32.49,0.10
1,2021,11,-0.50,0.87,313,Short,0.57,170.5,0.89,32.95,NaN,60417.64,19.36,0.22
2,2019,3,1.00,0.00,75,Long,0.99,133.3,1.29,54.55,69.0,93391.36,0.36,0.52


**📋 Engineered-feature documentation** *(reference answer; write your own, in your own words)*

| Feature | Formula | Why it should help |
|---|---|---|
| `plant_year` | year of planting | captures trend / year effects (e.g. prices, technology) |
| `plant_month`, `plant_month_sin`, `plant_month_cos` | month and its circular encoding | season; December is next to January |
| `plant_dayofyear` | day of year | finer position within the season |
| `rain_season` | Long (Mar–May) / Short (Oct–Dec) / Dry from planting month | rainy-season structure of Kenyan farming |
| `log_farm_size` | log(1 + farm size) | tames the heavy right-skew |
| `npk_total` | N + P + K | overall fertility |
| `n_to_p_ratio` | N / (P + 1) | nutrient balance (+1 guards zero) |
| `moisture_index` | rainfall / temperature | moisture available to the crop |
| `fert_x_irrigation` | fertiliser × irrigation flag | fertiliser pays off when water is available |
| `rain_gap_sq`, `temp_gap_sq` | (value − crop optimum)² | yields are hill-shaped around each crop's optimum |
| `ph_gap_sq` | (pH − 6.3)² | most crops prefer slightly acidic soil |

---
# Step 6 · Build the preprocessing pipeline  *(Lessons 3 and 7)*

**Goal:** one `Pipeline` object, `pipe`, that takes the **raw-ish** `X` (the output of `clean_farm` minus excluded columns) and ends in a `Ridge` model.

**Required structure:**
```
DateFeatures → AgriFeatures (your feature steps) → ColumnTransformer → SelectKBest → Ridge
```
**ColumnTransformer branches:**
- **numeric:** median imputation → outlier capping learned on train (Winsorizer) → standardisation
- **nominal** (`crop, county, soil_type, rain_season`): most-frequent imputation → one-hot (`handle_unknown='ignore'`)
- **ordinal** (`experience`, `pest_pressure`): most-frequent imputation → ordinal encoding with the **correct explicit order**

**Checkpoints:** the pipeline must `fit` on `X_train` and `predict` on `X_test` without errors, and the transformed training matrix must pass `check_ml_ready`.

In [10]:
NUM = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm", "organic_matter_pct",
       "fertilizer_kg_ha", "irrigation", "plant_year", "plant_month_sin", "plant_month_cos", "log_farm_size",
       "npk_total", "n_to_p_ratio", "moisture_index", "fert_x_irrigation", "rain_gap_sq", "temp_gap_sq", "ph_gap_sq"]
NOM = ["crop", "county", "soil_type", "rain_season"]
ORD = ["experience", "pest_pressure"]
ORD_CATS = [["Beginner", "Intermediate", "Advanced", "Expert"], ["Low", "Medium", "High"]]


def build_pipeline(model=None, k="all", strategy="median", cap_k=10.0, num=NUM):
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy=strategy)), ("cap", Winsorizer(k=cap_k)), ("scale", StandardScaler())]), num),
        ("nom", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), NOM),
        ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("ordinal", OrdinalEncoder(categories=ORD_CATS, handle_unknown="use_encoded_value", unknown_value=-1))]), ORD),
    ], remainder="drop").set_output(transform="pandas")
    return Pipeline([("dates", DateFeatures(harvest_col=None)), ("feats", AgriFeatures()), ("prep", prep),
                     ("select", SelectKBest(f_regression, k=k).set_output(transform="pandas")),
                     ("model", model if model is not None else Ridge(alpha=1.0))])


from sklearn import set_config
set_config(display="diagram")
pipe = build_pipeline()
pipe.fit(X_train, y_train)

Z_train = pipe[:-1].transform(X_train)               # what the model actually receives
print("matrix handed to the model:", Z_train.shape)
check_ml_ready(Z_train)
pipe

matrix handed to the model: (1866, 44)
  ✅ all columns numeric
  ✅ no missing values
  ✅ no infinite values
  ✅ no constant columns
  ✅ unique column names

5/5 checks passed  |  shape (1866, 44)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dates', ...), ('feats', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,harvest_col,None
,plant_col,'planting_date'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_thre

---
# Step 7 · Select features: with canaries  *(Lesson 5)*

**Goal:** decide how many features to keep, using evidence.

**Do this:**
1. Transform the training data with the preprocessing steps (fitted on train).
2. Append **3 random-noise "canary" columns** and compute mutual information with the target. The best canary score is your **noise floor**: which real features fall below it?
3. Use `GridSearchCV` over `select__k` (e.g. 10, 20, 30, `'all'`) with `GroupKFold(5)` to choose `k`.
4. State your decision in a sentence.

In [11]:
Zc = Z_train.copy()
rng = np.random.default_rng(0)
for i in range(3):
    Zc[f"canary_{i}"] = rng.normal(size=len(Zc))

MI = pd.Series(mutual_info_regression(Zc, y_train, random_state=0), index=Zc.columns)
canaries = [c for c in Zc.columns if c.startswith("canary")]
floor = MI[canaries].max()
below = [c for c in Z_train.columns if MI[c] <= floor]
print(f"noise floor (best canary MI) = {floor:.3f}")
print(f"{len(below)} of {Z_train.shape[1]} real features score at/below it, e.g.:", below[:8])

gs = GridSearchCV(build_pipeline(), {"select__k": [10, 20, 30, "all"]}, cv=GroupKFold(5), scoring="r2", n_jobs=-1)
gs.fit(X_train, y_train, groups=groups_train)
display(pd.DataFrame(gs.cv_results_)[["param_select__k", "mean_test_score"]].round(4))
BEST_K = gs.best_params_["select__k"]
print("chosen k:", BEST_K)

noise floor (best canary MI) = 0.011
22 of 44 real features score at/below it, e.g.: ['num__nitrogen_ppm', 'num__phosphorus_ppm', 'num__potassium_ppm', 'num__fertilizer_kg_ha', 'num__irrigation', 'num__plant_month_sin', 'num__npk_total', 'num__n_to_p_ratio']


,param_select__k,mean_test_score
0,10,0.9140
1,20,0.9329
2,30,0.9382
3,all,0.9393


chosen k: all


**📝 Decision (reference):** many one-hot county columns and a few weak numeric features fall near the noise floor, but removing them does not improve the cross-validated score; Ridge is robust to a few weak features. We therefore keep the value of `k` that scores best (often `'all'`), and rely on regularisation. *Selection is a tool, not an obligation.*

---
# Step 8 · Evaluate honestly  *(Lessons 6–7)*

**Goal:** show *quantitatively* that your preparation matters.

Compare with **5-fold `GroupKFold` cross-validation on the training set only** (R² on log-yield):

| Model | What it shows |
|---|---|
| Global mean (`DummyRegressor`) | the floor: R² ≈ 0 |
| Crop-only linear model | how much crop alone explains |
| Crude: raw numeric columns → mean-imputer → Ridge | "no preparation" |
| **Your pipeline → Ridge** | the value of Phase 2 |
| Your pipeline → Random Forest | does a stronger model change the story? |

Then run the **shuffled-target sanity test**: shuffle `y_train`, cross-validate your pipeline again. R² must collapse to ≈ 0. If it doesn't, something leaks.

In [12]:
cv = GroupKFold(5)
score = lambda model, data=X_train, target=y_train: cross_val_score(model, data, target, cv=cv, groups=groups_train, scoring="r2").mean()

raw_num = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "phosphorus_ppm", "potassium_ppm",
           "organic_matter_pct", "fertilizer_kg_ha", "farm_size_ha"]
crop_only = make_pipeline(ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), ["crop"])]), LinearRegression())

results = pd.Series({
    "global mean (floor)":                        score(DummyRegressor(), X_train[raw_num].fillna(0)),
    "crop only → linear":                         score(crop_only),
    "crude: raw numerics → mean impute → Ridge":  score(make_pipeline(SimpleImputer(strategy="mean"), Ridge()), X_train[raw_num]),
    "YOUR PIPELINE → Ridge":                      score(build_pipeline(k=BEST_K)),
    "YOUR PIPELINE → Random Forest":              score(build_pipeline(RandomForestRegressor(150, min_samples_leaf=3, n_jobs=-1, random_state=0), k=BEST_K)),
}, name="CV R² (log yield)")
display(results.round(3).to_frame())

# sanity test: a leak-free workflow cannot predict shuffled labels
shuffled = pd.Series(np.random.default_rng(0).permutation(y_train.values), index=y_train.index)
print("shuffled-target CV R²:", round(score(build_pipeline(k=BEST_K), X_train, shuffled), 3), "(should be ≈ 0 or slightly negative)")

,CV R² (log yield)
global mean (floor),-0.001
crop only → linear,0.644
crude: raw numerics → mean impute → Ridge,0.070
YOUR PIPELINE → Ridge,0.939
YOUR PIPELINE → Random Forest,0.930


shuffled-target CV R²: -0.032 (should be ≈ 0 or slightly negative)


---
# Step 9 · The final, one-shot test  *(Lesson 7)*

Fit the chosen pipeline on the **whole training set**, predict the **held-out farms once**, and report the result in units people understand (t/ha). Resist the urge to go back and tune after seeing this number: the test set has spoken once.

In [13]:
final = build_pipeline(k=BEST_K).fit(X_train, y_train)
pred, true = np.exp(final.predict(X_test)), np.exp(y_test)

from sklearn.metrics import r2_score, mean_absolute_error
print(f"held-out farms ({groups.iloc[te].nunique()}): R² = {r2_score(true, pred):.3f} | MAE = {mean_absolute_error(true, pred):.2f} t/ha")

by_crop = pd.DataFrame({"crop": X_test["crop"].values, "abs_error": np.abs(pred - true), "true": true}).groupby("crop").agg(
    n=("true", "size"), mean_yield=("true", "mean"), MAE=("abs_error", "mean")).round(2)
by_crop["MAE as % of mean yield"] = (by_crop["MAE"] / by_crop["mean_yield"] * 100).round(1)
by_crop

held-out farms (120): R² = 0.880 | MAE = 0.53 t/ha


,n,mean_yield,MAE,MAE as % of mean yield
crop,,,,
Beans,99,0.42,0.09,21.4
Maize,180,2.33,0.54,23.2
Potato,72,7.13,1.63,22.9
Sorghum,58,0.45,0.18,40.0
Wheat,54,1.58,0.27,17.1


---
# Step 10 · ⭐ Stretch goal: price context from the time-series data  *(Lesson 4, Part 3)*

Can *market context at planting time* help? Build `price_trend_30d` = the average daily price of the crop over the **30 days before planting** (Maize, Beans and Potato only; Wheat and Sorghum have no price series → NaN, handled by the imputer).

Rules: **trailing windows only**: nothing from the planting day or later. Test whether the feature improves cross-validated R²; report honestly either way.

In [14]:
def price_context(frame, window=30):
    """Mean crop price over the `window` days BEFORE planting (strictly past data)."""
    p = load_prices().sort_values(["crop", "date"]).reset_index(drop=True)
    spike = p["price_ksh_kg"] > 1.5 * p.groupby("crop")["price_ksh_kg"].transform(lambda s: s.shift(1).rolling(7, min_periods=3).median())
    p["price"] = p["price_ksh_kg"].mask(spike)
    p["price"] = p.groupby("crop")["price"].ffill()
    p["trail"] = p.groupby("crop")["price"].transform(lambda s: s.shift(1).rolling(window, min_periods=10).mean())   # shift(1): excludes today

    left = frame[["crop", "planting_date"]].reset_index().sort_values("planting_date")
    merged = pd.merge_asof(left, p[["date", "crop", "trail"]].dropna().sort_values("date"), left_on="planting_date",
                           right_on="date", by="crop", direction="backward")
    return merged.set_index("index")["trail"].reindex(frame.index)

X_plus = X_train.assign(price_trend_30d=price_context(X_train))
print("price context available for", int(X_plus["price_trend_30d"].notna().sum()), "of", len(X_plus), "rows (Maize/Beans/Potato)")

base = score(build_pipeline(k=BEST_K))
plus = score(build_pipeline(k=BEST_K, num=NUM + ["price_trend_30d"]), X_plus)
print(f"CV R² without: {base:.4f} | with price context: {plus:.4f} | change: {plus - base:+.4f}")

price context available for 1454 of 1866 rows (Maize/Beans/Potato)


CV R² without: 0.9393 | with price context: 0.9392 | change: -0.0001


**Reading the stretch result:** the change is essentially zero. That is an honest and useful finding: in this *simulated* data the farm's own recorded price does not follow the daily market series, so there is nothing to learn. In real data, price context might matter. What matters here is that you built a **time-safe** feature (`shift(1)` + trailing window + `merge_asof` backward) and *measured* it instead of assuming it helps.

---
# Step 11 · Export the deliverables 📦

| # | Deliverable | File |
|---|---|---|
| 1 | Cleaned dataset | `farm_clean.csv` *(saved in Step 2)* |
| 2 | Preprocessing pipeline (fitted on train) | `preprocessing_pipeline.joblib` (+ full model pipeline `yield_model_pipeline.joblib`) |
| 3 | Engineered features, ML-ready matrices | `X_train_ready.csv`, `X_test_ready.csv`, `y_train.csv`, `y_test.csv`, `feature_dictionary.csv` |
| 4 | Documented explanation of each transformation | `TRANSFORMATION_LOG.md` |

**Important:** the ML-ready matrices are produced by a pipeline **fitted on the training set only**; the test matrix is merely *transformed*.

In [15]:
# ---- 2. the pipelines
preprocess_only = final[:-1]                                   # dates -> features -> encode/scale -> select
joblib.dump(preprocess_only, OUT / "preprocessing_pipeline.joblib")
joblib.dump(final, OUT / "yield_model_pipeline.joblib")

# ---- 3. ML-ready matrices (fit on TRAIN, transform both)
X_train_ready = preprocess_only.transform(X_train)
X_test_ready = preprocess_only.transform(X_test)
X_train_ready.to_csv(OUT / "X_train_ready.csv", index=False)
X_test_ready.to_csv(OUT / "X_test_ready.csv", index=False)
y_train.rename("log_yield").to_csv(OUT / "y_train.csv", index=False)
y_test.rename("log_yield").to_csv(OUT / "y_test.csv", index=False)

# ---- feature dictionary: where does every final column come from?
engineered = set(new_cols)
rows = []
for name in X_train_ready.columns:
    branch, col = name.split("__", 1)
    rows.append({"feature": name,
                 "source": col,
                 "engineered": col in engineered,
                 "treatment": {"num": "median-impute -> cap (Winsorizer, fitted on train) -> standardise",
                               "nom": "mode-impute -> one-hot (unknown -> all zeros)",
                               "ord": "mode-impute -> ordinal with explicit order"}[branch]})
feature_dictionary = pd.DataFrame(rows)
feature_dictionary.to_csv(OUT / "feature_dictionary.csv", index=False)
print("exported matrices:", X_train_ready.shape, X_test_ready.shape, "| engineered columns:", int(feature_dictionary["engineered"].sum()))
feature_dictionary.head(8)

exported matrices: (1866, 44) (463, 44) | engineered columns: 11


,feature,source,engineered,treatment
0,num__rainfall_mm,rainfall_mm,False,"median-impute -> cap (Winsorizer, fitted on tr..."
1,num__avg_temp_c,avg_temp_c,False,"median-impute -> cap (Winsorizer, fitted on tr..."
2,num__soil_ph,soil_ph,False,"median-impute -> cap (Winsorizer, fitted on tr..."
3,num__nitrogen_ppm,nitrogen_ppm,False,"median-impute -> cap (Winsorizer, fitted on tr..."
4,num__phosphorus_ppm,phosphorus_ppm,False,"median-impute -> cap (Winsorizer, fitted on tr..."
5,num__potassium_ppm,potassium_ppm,False,"median-impute -> cap (Winsorizer, fitted on tr..."
6,num__organic_matter_pct,organic_matter_pct,False,"median-impute -> cap (Winsorizer, fitted on tr..."
7,num__fertilizer_kg_ha,fertilizer_kg_ha,False,"median-impute -> cap (Winsorizer, fitted on tr..."


### The transformation log
For **every** transformation: *what* it does, *why*, which columns, and whether it **learns from data** (the leakage question). This table is the most valuable part of your documentation: it is what lets a colleague trust and reproduce your work.

In [16]:
log = pd.DataFrame([
    ("1", "Type conversion",        "fertilizer_kg_ha, planting_date, harvest_date", "Extract numbers from text ('85 kg' -> 85); parse mixed date formats", "Models need numbers/dates, not text", "No"),
    ("2", "Category standardisation", "crop, county, soil_type, pest_pressure, irrigation", "lower-case, strip punctuation, synonyms (corn->Maize), fuzzy match; Yes/Y/TRUE/1 -> 1", "Dozens of spellings became 5 crops / 10 counties", "No (rules)"),
    ("3", "De-duplication",         "all rows", "Drop repeated (farm_id, planting_date), keep the most complete", "Avoid double-counting and train/test duplicates", "No"),
    ("4", "Cross-field logic",      "season, harvest_date", "Derive missing season from planting month; blank harvest < planting", "Deterministic repairs, no guessing", "No (rules)"),
    ("5", "Validity rules",         "soil_ph, rainfall, temperature, farm size, nutrients, fertilizer, price", "Out-of-range / sentinel values -> NaN", "Impossible values would mislead the model", "No (domain rules)"),
    ("6", "Target cleaning",        "yield_t_ha", "Drop missing, <=0 or above crop ceiling; model log(yield)", "Can't learn from impossible answers; log tames skew", "No"),
    ("7", "Leakage exclusions",     ", ".join(EXCLUDE), "Removed before modelling", "Not known at prediction time / identifiers / duplicates of other info", "No"),
    ("8", "Grouped split",          "rows", "80/20 split by farm_id", "Test on unseen farms", "No"),
    ("9", "DateFeatures",           "planting_date", "year, month, sin/cos month, day-of-year, rain season", "Seasonality", "No"),
    ("10", "AgriFeatures",          "N, P, K, rainfall, temp, pH, fertiliser, irrigation, farm size, crop", "npk_total, N:P ratio, moisture index, fert x irrigation, gap-to-optimum^2, log size", "Capture hill-shaped and balance effects", "No (row-wise)"),
    ("11", "Median imputation",     "numeric features", "Fill NaN with the column median", "Models can't take NaN; median robust to skew", "YES - medians (train only)"),
    ("12", "Winsorizer (k=10)",     "numeric features", "Clip to Q1-10*IQR / Q3+10*IQR; columns with IQR = 0 (binary flags) are left uncapped", "Limit extreme leverage (gentle: aggressive capping hurt; zero-IQR guard prevents erasing 0/1 columns)", "YES - fences (train only)"),
    ("13", "StandardScaler",        "numeric features", "(x - mean) / std", "Comparable scales for linear model", "YES - mean/std (train only)"),
    ("14", "Mode imputation + one-hot", "crop, county, soil_type, rain_season", "Fill with most frequent, then 0/1 columns; unseen -> zeros", "Nominal categories have no order", "YES - mode & vocabulary (train only)"),
    ("15", "Mode imputation + ordinal", "experience, pest_pressure", "Explicit order Beginner<...<Expert, Low<Medium<High", "Real order preserved", "Mode: YES; order: no"),
    ("16", "SelectKBest (k=%s)" % BEST_K, "all engineered/encoded columns", "Keep top-k by F-test inside the pipeline", "Reduce noise; evaluated by GroupKFold", "YES - scores (train only)"),
], columns=["#", "Step", "Columns", "What it does", "Why", "Learns from data?"])


def to_markdown(table):
    head = "| " + " | ".join(table.columns) + " |\n|" + "---|" * len(table.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v).replace("|", "/") for v in row) + " |" for row in table.itertuples(index=False))

clean_log = farm_clean.attrs["cleaning_log"]
doc = "# Transformation log; AgriReady pipeline\n\n"
doc += "**Use-case:** pre-harvest yield forecast (log yield). **Test:** 20% of farms, never used for fitting.\n\n"
doc += "## Cleaning counts\n\n" + to_markdown(pd.DataFrame(list(clean_log.items()), columns=["step", "count"])) + "\n\n"
doc += "## Every transformation\n\n" + to_markdown(log) + "\n\n"
doc += "## Leakage rule\n\nEvery step marked **YES** was fitted on the training set only and merely *applied* to the test set.\n"
(OUT / "TRANSFORMATION_LOG.md").write_text(doc, encoding="utf-8")
print("written:", OUT / "TRANSFORMATION_LOG.md", f"({len(doc):,} characters)")
log

written: TRANSFORMATION_LOG.md (3,535 characters)


,#,Step,Columns,What it does,Why,Learns from data?
0,1,Type conversion,"fertilizer_kg_ha, planting_date, harvest_date",Extract numbers from text ('85 kg' -> 85); par...,"Models need numbers/dates, not text",No
1,2,Category standardisation,"crop, county, soil_type, pest_pressure, irriga...","lower-case, strip punctuation, synonyms (corn-...",Dozens of spellings became 5 crops / 10 counties,No (rules)
2,3,De-duplication,all rows,"Drop repeated (farm_id, planting_date), keep t...",Avoid double-counting and train/test duplicates,No
3,4,Cross-field logic,"season, harvest_date",Derive missing season from planting month; bla...,"Deterministic repairs, no guessing",No (rules)
4,5,Validity rules,"soil_ph, rainfall, temperature, farm size, nut...",Out-of-range / sentinel values -> NaN,Impossible values would mislead the model,No (domain rules)
5,6,Target cleaning,yield_t_ha,"Drop missing, <=0 or above crop ceiling; model...",Can't learn from impossible answers; log tames...,No
6,7,Leakage exclusions,"farm_id, harvest_date, total_revenue_ksh, mark...",Removed before modelling,Not known at prediction time / identifiers / d...,No
7,8,Grouped split,rows,80/20 split by farm_id,Test on unseen farms,No
8,9,DateFeatures,planting_date,"year, month, sin/cos month, day-of-year, rain ...",Seasonality,No
9,10,AgriFeatures,"N, P, K, rainfall, temp, pH, fertiliser, irrig...","npk_total, N:P ratio, moisture index, fert x i...",Capture hill-shaped and balance effects,No (row-wise)


---
# Step 12 · Final verification ✅
Reload the saved pipeline in a *fresh* state and prove that it works on raw data.

In [17]:
print("Is the ML-ready training matrix model-ready?")
check_ml_ready(pd.read_csv(OUT / "X_train_ready.csv"))
print("\nIs the test matrix model-ready?")
check_ml_ready(pd.read_csv(OUT / "X_test_ready.csv"))

# reload and reuse on a messy new record
loaded = joblib.load(OUT / "yield_model_pipeline.joblib")
new = X_test.iloc[[0]].copy()
new["county"] = "Narok"; new["soil_ph"] = np.nan; new["pest_pressure"] = np.nan
print("\nreloaded pipeline, messy new record -> %.2f t/ha" % np.exp(loaded.predict(new))[0])
print("same predictions as the in-memory model:", np.allclose(loaded.predict(X_test), final.predict(X_test)))

print("\nfiles in", OUT.resolve().name + ":", sorted(p.name for p in OUT.iterdir() if p.is_file() and p.suffix in (".csv", ".joblib", ".md")))

Is the ML-ready training matrix model-ready?
  ✅ all columns numeric
  ✅ no missing values
  ✅ no infinite values
  ✅ no constant columns
  ✅ unique column names

5/5 checks passed  |  shape (1866, 44)

Is the test matrix model-ready?


  ✅ all columns numeric
  ✅ no missing values
  ✅ no infinite values
  ✅ no constant columns
  ✅ unique column names

5/5 checks passed  |  shape (463, 44)

reloaded pipeline, messy new record -> 1.39 t/ha
same predictions as the in-memory model: True

files in reference_solution: ['TRANSFORMATION_LOG.md', 'X_test_ready.csv', 'X_train_ready.csv', 'farm_clean.csv', 'feature_dictionary.csv', 'preprocessing_pipeline.joblib', 'y_test.csv', 'y_train.csv', 'yield_model_pipeline.joblib']


---
# 🪞 Reflection (model answers in the course docs)

1. Which **single cleaning decision** changed your results the most? How do you know?
2. Name **two features** you excluded for leakage reasons and explain the mechanism in each case.
3. Why did you split **by farm** rather than by random row?
4. Which transformations learn from data? What would have gone wrong if you fitted them on all rows?
5. Where did your pipeline behave **worst** (which crop / which kind of farm)? What would you do next?
6. If the county office wanted a forecast **at planting time** instead of before harvest, what in your pipeline would you change?

---
# ✅ Submission checklist
- [ ] `farm_clean.csv` · `preprocessing_pipeline.joblib` · `yield_model_pipeline.joblib`
- [ ] `X_train_ready.csv`, `X_test_ready.csv`, `y_train.csv`, `y_test.csv`, `feature_dictionary.csv`
- [ ] `TRANSFORMATION_LOG.md` (every transformation: what, why, learns-from-data?)
- [ ] `check_clean` 13/13 and `check_ml_ready` 5/5
- [ ] Shuffled-target test ≈ 0; leakage exclusions documented
- [ ] This notebook runs top-to-bottom without errors